# Feature Engineering

This notebook starts from the cleaned Airbnb listings dataset created in Part 1.

Input:
- `data/processed/listings_cleaned.csv`

Output:
- `data/processed/modeling_dataset.csv`

The goal is to prepare a complete numeric table for Part 3 while keeping the existing feature-engineering workflow easy to inspect. Part 3 reconstructs relevant originally missing numeric values from the accompanying indicators and fits imputation on training data only.


In [1]:
import pandas as pd
import numpy as np

In [2]:
listings_cleaned = pd.read_csv("../data/processed/listings_cleaned.csv")
print("Shape:", listings_cleaned.shape)

Shape: (9171, 97)


In [3]:
listings_cleaned.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_checkin_was_missing,review_scores_communication_was_missing,review_scores_location_was_missing,review_scores_value_was_missing,host_response_rate_was_missing,host_acceptance_rate_was_missing,reviews_per_month_was_missing,bathrooms_parsed_was_missing,bedrooms_was_missing,beds_was_missing
0,3176,https://www.airbnb.com/rooms/3176,20250923202926,2025-09-24,city scrape,Fabulous Flat in great Location,This beautiful first floor apartment is situa...,The neighbourhood is famous for its variety of...,https://a0.muscache.com/pictures/hosting/Hosti...,3718,...,0,0,0,0,0,0,0,0,0,0
1,9991,https://www.airbnb.com/rooms/9991,20250923202926,2025-09-24,city scrape,Geourgeous flat - outstanding views,4 bedroom with very large windows and outstand...,Prenzlauer Berg is an amazing neighbourhood wh...,https://a0.muscache.com/pictures/42799131/59c8...,33852,...,0,0,0,0,0,0,0,0,0,0
2,14325,https://www.airbnb.com/rooms/14325,20250923202926,2025-09-24,city scrape,Studio Apartment in Prenzlauer Berg,The apartment is located on the upper second f...,NaN,https://a0.muscache.com/pictures/508703/24988a...,55531,...,0,0,0,0,0,0,0,0,0,0
3,17904,https://www.airbnb.com/rooms/17904,20250923202926,2025-09-24,city scrape,Beautiful Kreuzberg studio - 3 months minimum,"- apt is available starting October 1, 2025 (m...","The apartment is located in Kreuzberg, which i...",https://a0.muscache.com/pictures/d9a6f8be-54b9...,68997,...,0,0,0,0,0,0,0,0,0,0
4,20858,https://www.airbnb.com/rooms/20858,20250923202926,2025-09-24,city scrape,Designer Loft in Berlin Mitte,Bright and sunny condo with two balconies in a...,Fantastic vibe in the middle of the popular Ka...,https://a0.muscache.com/pictures/108232/205b19...,71331,...,0,0,0,0,0,0,0,0,0,0


In [4]:
listings_cleaned.columns.tolist()

['id',
 'listing_url',
 'scrape_id',
 'last_scraped',
 'source',
 'name',
 'description',
 'neighborhood_overview',
 'picture_url',
 'host_id',
 'host_url',
 'host_name',
 'host_since',
 'host_location',
 'host_about',
 'host_response_time',
 'host_response_rate',
 'host_acceptance_rate',
 'host_is_superhost',
 'host_thumbnail_url',
 'host_picture_url',
 'host_neighbourhood',
 'host_listings_count',
 'host_total_listings_count',
 'host_verifications',
 'host_has_profile_pic',
 'host_identity_verified',
 'neighbourhood',
 'neighbourhood_cleansed',
 'neighbourhood_group_cleansed',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bathrooms',
 'bathrooms_text',
 'bedrooms',
 'beds',
 'amenities',
 'price',
 'minimum_nights',
 'maximum_nights',
 'minimum_minimum_nights',
 'maximum_minimum_nights',
 'minimum_maximum_nights',
 'maximum_maximum_nights',
 'minimum_nights_avg_ntm',
 'maximum_nights_avg_ntm',
 'calendar_updated',
 'has_availability',
 'availability_30

Select useful columns for prediction

In [5]:
selected_columns = [
    "id",
    
    # Target
    "price",

    # Basic listing information (listing features)
    "room_type",
    "accommodates",
    "bathrooms_parsed",
    "bedrooms",
    "beds",

    # Booking rules and availability (booking features)
    "minimum_nights",
    "maximum_nights",
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",

    # Review information (quality features)
    "number_of_reviews",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "reviews_per_month",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",

    # Location information (spatial features)
    "latitude",
    "longitude",
    "neighbourhood_group_cleansed",

    # Host-related simple features
    "host_is_superhost",
    "host_identity_verified",
    "instant_bookable",
    "host_listings_count",
    "host_total_listings_count",

    # Text columns (text features)
    "name",
    "description",
    "neighborhood_overview",

    # Missing-value indicator columns created in Part 1
    "host_since_was_missing",
    "first_review_was_missing",
    "last_review_was_missing",
    "review_scores_rating_was_missing",
    "review_scores_accuracy_was_missing",
    "review_scores_cleanliness_was_missing",
    "review_scores_checkin_was_missing",
    "review_scores_communication_was_missing",
    "review_scores_location_was_missing",
    "review_scores_value_was_missing",
    "host_response_rate_was_missing",
    "host_acceptance_rate_was_missing",
    "reviews_per_month_was_missing",
    "bathrooms_parsed_was_missing",
    "bedrooms_was_missing",
    "beds_was_missing"
]

modeling_data = listings_cleaned[selected_columns].copy()

print("Shape of selected modeling data:", modeling_data.shape)
modeling_data.head()

Shape of selected modeling data: (9171, 51)


,id,price,room_type,accommodates,bathrooms_parsed,bedrooms,beds,minimum_nights,maximum_nights,availability_30,...,review_scores_checkin_was_missing,review_scores_communication_was_missing,review_scores_location_was_missing,review_scores_value_was_missing,host_response_rate_was_missing,host_acceptance_rate_was_missing,reviews_per_month_was_missing,bathrooms_parsed_was_missing,bedrooms_was_missing,beds_was_missing
0,3176,105.0,Entire home/apt,2,1.0,1.0,2.0,63,730,0,...,0,0,0,0,0,0,0,0,0,0
1,9991,135.0,Entire home/apt,7,2.5,4.0,4.0,6,14,0,...,0,0,0,0,0,0,0,0,0,0
2,14325,75.0,Entire home/apt,1,1.0,0.0,1.0,150,1125,17,...,0,0,0,0,0,0,0,0,0,0
3,17904,32.0,Entire home/apt,2,1.0,0.0,1.0,93,360,11,...,0,0,0,0,0,0,0,0,0,0
4,20858,202.0,Entire home/apt,4,1.0,2.0,2.0,3,21,15,...,0,0,0,0,0,0,0,0,0,0


Check & handle missing values

In [6]:
# Check missing values in the selected data

missing_values = modeling_data.isna().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
missing_values

neighborhood_overview        5045
description                   340
host_identity_verified          2
host_total_listings_count       2
host_listings_count             2
dtype: int64

In [7]:
# Fill the small number of remaining values needed for a complete numeric export.
# Part 3 reconstructs relevant originally missing numeric values and performs
# training-only imputation inside the modeling pipelines.
modeling_data["description"] = modeling_data["description"].fillna("")
modeling_data["neighborhood_overview"] = modeling_data["neighborhood_overview"].fillna("")
modeling_data["host_identity_verified"] = modeling_data["host_identity_verified"].fillna("unknown")
modeling_data["host_listings_count"] = modeling_data["host_listings_count"].fillna(
    modeling_data["host_listings_count"].median()
)
modeling_data["host_total_listings_count"] = modeling_data["host_total_listings_count"].fillna(
    modeling_data["host_total_listings_count"].median()
)


Spatial features


A new spatial feature is created using latitude and longitude: the distance of each listing to Berlin city center. 

In [8]:
def haversine_distance(lat1, lon1, lat2=52.5200, lon2=13.4050):
    """
    Calculate distance between two geographic points.
    
    Default destination:
    Berlin city center, approximately 52.5200 latitude and 13.4050 longitude.
    
    Returns distance in kilometers.
    """
    R = 6371  #Earth radius in kilometers

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = ( np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2)
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c


modeling_data["distance_to_center_km"] = haversine_distance(modeling_data["latitude"], modeling_data["longitude"])
modeling_data[["latitude", "longitude", "distance_to_center_km"]].head()

,latitude,longitude,distance_to_center_km
0,52.53471,13.41810,1.860321
1,52.53269,13.41805,1.664484
2,52.54813,13.40366,3.129226
3,52.49419,13.42166,3.083494
4,52.53711,13.40888,1.920565


Text features

Simple numerical text features are created from the listing name, description, and neighborhood overview. These features capture how much textual information is provided for each listing.


In [9]:
modeling_data["name_length"] = modeling_data["name"].str.len()
modeling_data["name_word_count"] = modeling_data["name"].str.split().str.len()

modeling_data["description_length"] = modeling_data["description"].str.len()
modeling_data["description_word_count"] = modeling_data["description"].str.split().str.len()

modeling_data["neighborhood_overview_length"] = modeling_data["neighborhood_overview"].str.len()
modeling_data["neighborhood_overview_word_count"] = modeling_data["neighborhood_overview"].str.split().str.len()

modeling_data[
    [
        "name",
        "name_length",
        "name_word_count",
        "description_length",
        "description_word_count",
        "neighborhood_overview_length",
        "neighborhood_overview_word_count"
    ]
].head()

,name,name_length,name_word_count,description_length,description_word_count,neighborhood_overview_length,neighborhood_overview_word_count
0,Fabulous Flat in great Location,31,5,315,51,509,82
1,Geourgeous flat - outstanding views,35,5,246,39,328,54
2,Studio Apartment in Prenzlauer Berg,35,5,436,73,0,0
3,Beautiful Kreuzberg studio - 3 months minimum,45,7,530,95,550,100
4,Designer Loft in Berlin Mitte,29,5,93,14,187,32


In [10]:
modeling_data = modeling_data.drop(columns=["name", "description", "neighborhood_overview"])
modeling_data.head()

,id,price,room_type,accommodates,bathrooms_parsed,bedrooms,beds,minimum_nights,maximum_nights,availability_30,...,bathrooms_parsed_was_missing,bedrooms_was_missing,beds_was_missing,distance_to_center_km,name_length,name_word_count,description_length,description_word_count,neighborhood_overview_length,neighborhood_overview_word_count
0,3176,105.0,Entire home/apt,2,1.0,1.0,2.0,63,730,0,...,0,0,0,1.860321,31,5,315,51,509,82
1,9991,135.0,Entire home/apt,7,2.5,4.0,4.0,6,14,0,...,0,0,0,1.664484,35,5,246,39,328,54
2,14325,75.0,Entire home/apt,1,1.0,0.0,1.0,150,1125,17,...,0,0,0,3.129226,35,5,436,73,0,0
3,17904,32.0,Entire home/apt,2,1.0,0.0,1.0,93,360,11,...,0,0,0,3.083494,45,7,530,95,550,100
4,20858,202.0,Entire home/apt,4,1.0,2.0,2.0,3,21,15,...,0,0,0,1.920565,29,5,93,14,187,32


Encode categorical variables

Categorical variables are converted into numeric variables using one-hot encoding.

In [11]:
# Encode categorical variables using one-hot encoding

binary_columns = [
    "host_is_superhost",
    "host_identity_verified",
    "instant_bookable"
]

for col in binary_columns:
    modeling_data[col] = modeling_data[col].map({
        "t": 1,
        "f": 0,
        "unknown": 0,
        "Unknown": 0
    }).fillna(0).astype(int)

categorical_columns = [
    "room_type",
    "neighbourhood_group_cleansed"
]

modeling_data_encoded = pd.get_dummies(
    modeling_data,
    columns=categorical_columns,
    drop_first=True
)

bool_columns = modeling_data_encoded.select_dtypes(include="bool").columns
modeling_data_encoded[bool_columns] = modeling_data_encoded[bool_columns].astype(int)

assert all(
    set(modeling_data_encoded[col].dropna().unique()).issubset({0, 1})
    for col in binary_columns
), "Binary encoding produced values outside {0, 1}."

print("Shape before encoding:", modeling_data.shape)
print("Shape after encoding:", modeling_data_encoded.shape)

modeling_data_encoded.head()


Shape before encoding: (9171, 55)
Shape after encoding: (9171, 67)


,id,price,accommodates,bathrooms_parsed,bedrooms,beds,minimum_nights,maximum_nights,availability_30,availability_60,...,neighbourhood_group_cleansed_Lichtenberg,neighbourhood_group_cleansed_Marzahn - Hellersdorf,neighbourhood_group_cleansed_Mitte,neighbourhood_group_cleansed_Neukölln,neighbourhood_group_cleansed_Pankow,neighbourhood_group_cleansed_Reinickendorf,neighbourhood_group_cleansed_Spandau,neighbourhood_group_cleansed_Steglitz - Zehlendorf,neighbourhood_group_cleansed_Tempelhof - Schöneberg,neighbourhood_group_cleansed_Treptow - Köpenick
0,3176,105.0,2,1.0,1.0,2.0,63,730,0,0,...,0,0,0,0,1,0,0,0,0,0
1,9991,135.0,7,2.5,4.0,4.0,6,14,0,1,...,0,0,0,0,1,0,0,0,0,0
2,14325,75.0,1,1.0,0.0,1.0,150,1125,17,17,...,0,0,0,0,1,0,0,0,0,0
3,17904,32.0,2,1.0,0.0,1.0,93,360,11,11,...,0,0,0,1,0,0,0,0,0,0
4,20858,202.0,4,1.0,2.0,2.0,3,21,15,43,...,0,0,0,0,1,0,0,0,0,0


Annotation table

In [12]:
def describe_column(column):
    if column == "id":
        return "Identifier", "Unique Airbnb listing ID. Kept for reference, not used for model training.", "ID"

    if column == "price":
        return "Target", "Nightly listing price to predict.", "currency (euro)"

    if column in ["accommodates", "bathrooms_parsed", "bedrooms", "beds"]:
        return "Listing feature", "Basic property capacity or room information.", "count"

    if column in ["minimum_nights", "maximum_nights"]:
        return "Booking rule", "Minimum or maximum number of nights allowed for booking.", "days"

    if column.startswith("availability_"):
        return "Availability", "Number of days the listing is available in the given time window.", "days"

    if column.startswith("number_of_reviews"):
        return "Review feature", "Number of reviews received by the listing.", "count"

    if column == "reviews_per_month":
        return "Review feature", "Average number of reviews per month.", "reviews/month"

    if column.startswith("review_scores_"):
        return "Review score", "Guest review score for this category.", "score"

    if column in ["latitude", "longitude"]:
        return "Spatial feature", "Geographic coordinate of the listing.", "coordinate"

    if column == "distance_to_center_km":
        return "Spatial feature", "Distance from listing to Berlin city center.", "kilometers"

    if column in ["host_listings_count", "host_total_listings_count"]:
        return "Host feature", "Number of listings associated with the host.", "count"

    if column in ["host_is_superhost", "host_identity_verified", "instant_bookable"]:
        return "Binary feature", "Binary yes/no feature. 1 means yes, 0 means no.", "0/1"

    if column.endswith("_was_missing"):
        return "Missing indicator", "Shows whether the original value was missing. 1 means missing, 0 means not missing.", "0/1"

    if column.endswith("_length"):
        return "Text feature", "Number of characters in the original text field.", "characters"

    if column.endswith("_word_count"):
        return "Text feature", "Number of words in the original text field.", "words"

    if column.startswith("room_type_"):
        return "Encoded categorical feature", "One-hot encoded room type. 1 means yes, 0 means no.", "0/1"

    if column.startswith("neighbourhood_group_cleansed_"):
        return "Encoded categorical feature", "One-hot encoded Berlin neighbourhood group. 1 means yes, 0 means no.", "0/1"

    return "Other", "Feature used for modeling.", "numeric"


feature_description = pd.DataFrame([
    {
        "column": column,
        "feature_group": describe_column(column)[0],
        "description": describe_column(column)[1],
        "unit_or_type": describe_column(column)[2]
    }
    for column in modeling_data_encoded.columns
])

feature_description.head()

,column,feature_group,description,unit_or_type
0,id,Identifier,"Unique Airbnb listing ID. Kept for reference, ...",ID
1,price,Target,Nightly listing price to predict.,currency (euro)
2,accommodates,Listing feature,Basic property capacity or room information.,count
3,bathrooms_parsed,Listing feature,Basic property capacity or room information.,count
4,bedrooms,Listing feature,Basic property capacity or room information.,count


In [13]:
output_path = "../data/processed/modeling_dataset.csv"

# Final integrity checks before saving the model-ready table.
assert modeling_data_encoded["id"].is_unique, "Listing IDs must be unique."
assert modeling_data_encoded["price"].notna().all(), "The target contains missing values."
assert (modeling_data_encoded["price"] > 0).all(), "Prices must be positive."
assert modeling_data_encoded.isna().sum().sum() == 0, "The modeling table still contains missing values."
assert np.isfinite(
    modeling_data_encoded.select_dtypes(include=np.number).to_numpy()
).all(), "The modeling table contains infinite numeric values."

expected_binary_prefixes = ("room_type_", "neighbourhood_group_cleansed_")
encoded_indicator_columns = [
    c for c in modeling_data_encoded.columns
    if c in binary_columns
    or c.endswith("_was_missing")
    or c.startswith(expected_binary_prefixes)
]
assert all(
    set(modeling_data_encoded[c].dropna().unique()).issubset({0, 1})
    for c in encoded_indicator_columns
), "At least one indicator column is not binary."

modeling_data_encoded.to_csv(output_path, index=False)
print(
    f"Saved {modeling_data_encoded.shape[0]:,} rows and "
    f"{modeling_data_encoded.shape[1]} columns to {output_path}"
)


Saved 9,171 rows and 67 columns to ../data/processed/modeling_dataset.csv
